In [11]:
import absl.logging
import logging
import requests
import json
import pandas as pd
import os
from datetime import datetime
import io
import time

from download_pdfs import download_pdfs
from grader import grade_paper
from parse import parse_pdfs_in_directory

logging.basicConfig(filename='research_helper.log', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(filename)s - %(funcName)s - %(message)s')

def search_and_save(query, fields, filename):
    """
    Searches Semantic Scholar API with a given query and saves results to a JSONL file
    in a subdirectory with a timestamp.

    Args:
      query: The search query.
      fields: The fields to retrieve from the API.
      filename: The name of the file to save the results.
    """
    try:
        logging.debug(f"Starting search_and_save for query: {query}")
        # Create subdirectories if they don't exist
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_dir = os.path.join("query_jsons", timestamp)
        os.makedirs(output_dir, exist_ok=True)

        filepath = os.path.join(output_dir, filename)

        url = f"http://api.semanticscholar.org/graph/v1/paper/search/bulk?query={query}&fields={fields}&year=-2025"
        logging.debug(f"Sending initial request to Semantic Scholar API: {url}")

        response = requests.get(url)

        logging.debug(f"Response status code: {response.status_code}")  # Log the status code
        logging.debug(f"Response headers: {response.headers}")

        response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
        r = response.json()

        logging.debug(f"Response JSON: {r}")

        logging.info(f"Estimated documents for query '{query}': {r.get('total', 'unknown')}")

        retrieved = 0
        with open(filepath, "a") as file:
            while True:
                if "data" in r:
                    retrieved += len(r["data"])
                    logging.debug(f"Retrieved {retrieved} papers for query '{query}'...")
                    for paper in r["data"]:
                        print(json.dumps(paper), file=file)
                if "token" not in r:
                    break

                time.sleep(10)
                
                response = requests.get(f"{url}&token={r['token']}")
                response.raise_for_status()
                r = response.json()

        logging.info(f"Retrieved {retrieved} papers total for query: {query}")

    except requests.exceptions.RequestException as e:
        logging.error(f"Error searching for query '{query}': {e}")
    except Exception as e:
        logging.exception(f"An unexpected error occurred while searching for query '{query}'")


def extrude_external_ids(df):
    """
    Extrudes the content of the 'externalIds' column in a DataFrame.

    Args:
      df: The input DataFrame with an 'externalIds' column containing dictionaries.

    Returns:
      A new DataFrame with the 'externalIds' content extruded into separate columns.
    """
    try:
        external_ids_df = df['externalIds'].apply(pd.Series)
        return pd.concat([df, external_ids_df], axis=1).drop(columns=['externalIds'])
    except Exception as e:
        logging.exception("An unexpected error occurred while extruding external IDs")
        raise  # Re-raise the exception after logging

def extract_urls(df):
    """
    Extracts all URLs from the 'openAccessPdf' column in a DataFrame.

    Args:
      df: The input DataFrame with an 'openAccessPdf' column containing dictionaries.

    Returns:
      A list of URLs extracted from the 'openAccessPdf' column.
    """
    try:
        return df['openAccessPdf'].apply(lambda x: x.get('url')).tolist()
    except Exception as e:
        logging.exception("An unexpected error occurred while extracting URLs")
        raise  # Re-raise the exception after logging

def process_and_download(user_query, queries, output_filename="merged_results.pkl"):
    """
    Processes a list of queries, merges the results, downloads the PDFs, and grades the papers.

    Args:
      user_query: The original user query.
      queries: A list of queries.
      output_filename: The name of the file to save the merged DataFrame.
    """
    try:
        fields = "title,year,abstract,externalIds,url,isOpenAccess,openAccessPdf,fieldsOfStudy"
        dfs = []

        # Create subdirectories if they don't exist
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_dir = os.path.join("query_jsons", timestamp)
        os.makedirs(output_dir, exist_ok=True)

        for query in queries:
            filename = query.lower().replace(" ", "_") + ".jsonl"
            filepath = os.path.join(output_dir, filename)
            search_and_save(query, fields, filepath)
            df = pd.read_json(io.StringIO(filepath), lines=True)
            dfs.append(df)

        merged_df = pd.concat(dfs, axis=0)

        # Grade the papers
        logging.info("Grading papers for relevance...")
        merged_df['relevance_grade'] = merged_df['abstract'].apply(
            lambda abstract: grade_paper(user_query, abstract)
        )
        logging.info("Finished grading papers.")

        # Filter for relevant papers (Apply the mask here)
        merged_df = merged_df[merged_df['relevance_grade'] == True]

        merged_df = extrude_external_ids(merged_df)  # Extrude IDs after filtering
        merged_df.to_pickle(output_filename)
        logging.info(f"Merged DataFrame saved to {output_filename}")

        filtered = merged_df[merged_df["abstract"].notnull()]
        open_access = filtered[filtered["isOpenAccess"] == True]
        final_df = open_access[open_access["openAccessPdf"].notnull()]

        urls = extract_urls(final_df)
        download_pdfs(urls, "pdfs", final_df)

        parse_pdfs_in_directory()

    except Exception as e:
        logging.exception("An unexpected error occurred while processing and downloading")


if __name__ == "__main__":
    absl.logging.set_verbosity(absl.logging.ERROR)
    import sys
    if len(sys.argv) < 2:
        # Log the usage message without the user query
        logging.info("Usage: python search.py <query> [output_filename]")  
        sys.exit(1)

    user_query = sys.argv[1]
    output_filename = sys.argv[2] if len(sys.argv) > 2 else "merged_results.pkl"
    
    # Log that a search is starting
    logging.info("Starting search process...")  

    from queries import query_chain
    results = query_chain.invoke({"query": user_query})
    process_and_download(results.queries, output_filename)

In [12]:
# Call the process_and_download function with a test query
user_query = "neanderthal and carnivores relationship during pleistocene"
queries = ["neanderthal", "carnivores pleistocene"]  # Example sub-queries, or generate using query_chain
process_and_download(user_query, queries, "test_results.pkl")

In [ ]:
import pymupdf4llm
import pathlib

md_text = pymupdf4llm.to_markdown("./pdfs/The grassiness of all flesh.pdf")
pathlib.Path("pymuf.md").write_bytes(md_text.encode())

Processing ./pdfs/The grassiness of all flesh.pdf...
[                                        ] (0/46[                                        ] ( 1/46[=                                       ] ( 2/4[==                                      ] ( 3/46[===                                     ] ( 4/4[====                                    ] ( 5/46[=====                                   ] ( 6/4[======                                  ] ( 7/4[======                                  ] ( 8/46[=======                                 ] ( 9/4[========                                ] (10/46[=========                               ] (11/4[==========                              ] (12/46[===========                             ] (13/4[============                            ] (14/46[=============                           ] (15/[=============                           ] (16/4[==============                          ] (17/46[===============                         ] (18/4[================           

In [ ]:
md_text

In [6]:
import pathlib
pathlib.Path("pymuf.md").write_bytes(md_text.encode())

141616

In [1]:
# Use MegaParse
from megaparse import MegaParse
from megaparse.parser.unstructured_parser import UnstructuredParser
import nest_asyncio
nest_asyncio.apply()

parser = UnstructuredParser()
megaparse = MegaParse(parser)
response = megaparse.load("./pdfs/The grassiness of all flesh.pdf")
megaparse.save("./megaparse.md")

/Users/asdls/miniconda3/envs/megaparse/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in UploadFileConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/asdls/miniconda3/envs/megaparse/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at microsoft/table-transformer-structure-recognition were not used when initializing TableTransformerForObjectDetection: ['model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expe

In [1]:
# store_into_db.py
import pandas as pd
import sqlite3
import logging
from db_manager import store_paper_data  # your aggregator store function

df = pd.read_pickle("merged_results.pkl")
papers = df.to_dict("records")

print("Number of rows to store: %d", len(papers))
store_paper_data(papers)

# Then you can check if the DB is still 0 bytes


Number of rows to store: %d 1400


In [3]:
papers[:10]


[{'title': 'Alternating carnivore and Neanderthal activities at Escoural Cave: insights from the taphonomic and machine learning analysis of leporid remains',
  'abstract': 'Exploring the varied subsistence strategies and cave occupation patterns of Neanderthals is key to understanding their complex behaviors and ecological adaptations. Small game consumption, in particular, is considered a relevant indicator of their behavioral complexity. Rabbit assemblages from Pleistocene cave sites provide valuable insights into Neanderthal interactions with small prey and potential competition with carnivores. Here, we present the first detailed taphonomic analysis of faunal remains from Escoural Cave (Portugal), where a European rabbit (Oryctolagus cuniculus) assemblage was found alongside Middle Paleolithic stone tools and some macromammal remains. This study combines traditional zooarchaeological and taphonomic analysis of the rabbit remains with multivariate statistics and machine learning me

In [4]:
df["DOI"].value_counts()

DOI
tempdoi-765bafdc08d4                8
10.3389/fearc.2024.1473266          6
10.4467/00015229aac.21.002.15343    6
tempdoi-a8e848da580b                6
10.1101/2020.04.19.049734           6
                                   ..
10.24377/LJMU.T.00015514            1
10.1002/ajpa.24926                  1
tempdoi-d43510400967                1
tempdoi-fe8eaf7f080f                1
10.1007/s12520-024-02026-0          1
Name: count, Length: 687, dtype: int64